# HMDB51 VideoMAE V2 Distilled Fine-Tuning — v7

This notebook fine-tunes an official OpenGVLab **VideoMAE V2 Kinetics-710 distilled checkpoint** on extracted HMDB51 frames while retaining the split, duplicate, checkpoint, and test-isolation protections from the audited MViT pipeline.

The default is **ViT-S distilled**, which is the safer first run on a typical Colab GPU. Change one setting, `VIDEOMAE_VARIANT = 'base'`, to use the larger ViT-B checkpoint. Test evaluation is disabled by default while the architecture is being selected.


## Experiment design

The primary comparison reuses the exact MViT `splits.json` when it is available. That keeps train, validation, and test membership fixed across backbones. Model selection and early stopping use validation only; the held-out test set remains a separate, explicit action.

VideoMAE V2 is a stronger **pretraining path**, not a guaranteed accuracy increase. The notebook therefore includes a verified checkpoint preflight, a one-epoch GPU smoke test, optional fixed-split multi-seed runs, and an optional ViT-B switch.


## 1. Mount Google Drive


In [1]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


## 2. Configuration


In [2]:
from pathlib import Path

# -----------------------------
# Primary selection
# -----------------------------
VIDEOMAE_VARIANT = 'small'  # 'small' or 'base'
SEED = 42
SPLIT_SEED = 42
MULTI_SEED_VALUES = (42, 123, 2026)

if VIDEOMAE_VARIANT == 'small':
    ARCHITECTURE = 'videomaev2_vit_s_distilled'
    BATCH_SIZE = 2
    GRADIENT_ACCUMULATION_STEPS = 4
    DROP_PATH_RATE = 0.10
elif VIDEOMAE_VARIANT == 'base':
    ARCHITECTURE = 'videomaev2_vit_b_distilled'
    BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 8
    DROP_PATH_RATE = 0.15
else:
    raise ValueError("VIDEOMAE_VARIANT must be 'small' or 'base'")

# -----------------------------
# Google Drive paths
# -----------------------------
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / 'Homework_4'
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = (
    PROJECT_ROOT
    / f'results_videomaev2_v7_{VIDEOMAE_VARIANT}_seed{SEED}'
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MULTI_SEED_ROOT = (
    PROJECT_ROOT
    / f'results_videomaev2_v7_{VIDEOMAE_VARIANT}_multiseed'
)

REPO_ZIP = PROJECT_ROOT / 'hmdb51_repo_videomaev2_distilled_v7.zip'
REPO_EXTRACT_ROOT = Path('/content/hmdb51_repo_videomaev2_distilled_v7')
REPO_DIR = REPO_EXTRACT_ROOT

DATASET_ZIP = PROJECT_ROOT / 'HMDB51.zip'
DATASET_EXTRACT_ROOT = Path('/content/hmdb51_data')
DATASET_DIR = None

# Reuse this exact split for the fairest MViT-versus-VideoMAE comparison.
REFERENCE_MVIT_SPLIT = (
    PROJECT_ROOT
    / 'results_mvit_generalization_v6_seed42'
    / 'splits.json'
)
REUSE_MVIT_SPLIT_IF_AVAILABLE = True

# -----------------------------
# Official HMDB51 split protocol
# -----------------------------
SPLIT_PROTOCOL = 'official'
OFFICIAL_SPLIT_DIR = PROJECT_ROOT / 'testTrainMulti_7030_splits'
OFFICIAL_SPLIT_NUMBER = 1
OFFICIAL_VALIDATION_SIZE = 0.15
FINGERPRINT_SPLIT_AUDIT = True
OFFICIAL_DUPLICATE_POLICY = 'drop_train'  # error, drop_train, or allow

AUTO_DOWNLOAD_OFFICIAL_SPLITS = True
OFFICIAL_SPLIT_HF_REPO = 'Serrelab/hmdb51'
OFFICIAL_SPLIT_HF_REVISION = '1f1e6df3fccdfe34a230e60c00a799b82905b35f'
OFFICIAL_SPLIT_ARCHIVE_NAME = 'test_train_splits.rar'
OFFICIAL_SPLIT_ARCHIVE = PROJECT_ROOT / OFFICIAL_SPLIT_ARCHIVE_NAME
OFFICIAL_SPLIT_ARCHIVE_MD5 = '15e67781e70dcfbdce2d7dbb9b3344b5'
OFFICIAL_SPLIT_ARCHIVE_SHA256 = (
    '229c94f845720d01eb3946d39f39292ea962d50a18136484aa47c1eba251d2b7'
)
OFFICIAL_SPLIT_EXTRACT_ROOT = Path('/content/hmdb51_official_split_extract')

# -----------------------------
# Official VideoMAE V2 weights
# -----------------------------
PRETRAINED_CACHE_DIR = PROJECT_ROOT / 'videomaev2_pretrained_cache'
LOCAL_PRETRAINED_CHECKPOINT = None  # Optional Path to the exact pinned file.
PREFETCH_PRETRAINED_WEIGHTS = True
RUN_CHECKPOINT_PREFLIGHT = True
VERIFY_PRETRAINED_SHA256 = True
GRADIENT_CHECKPOINTING = True
EVAL_CLIP_CHUNK_SIZE = 1
HEAD_INIT_SCALE = 0.001

# -----------------------------
# Model and temporal sampling
# -----------------------------
N_CLASSES = 51
FRAMES_PER_VIDEO = 16
IMAGE_SIZE = 224
EXTRACTED_FPS = 30.0  # Replace with the rate represented by adjacent frame files.
PRETRAINED_EVAL_FPS = 7.5
TEMPORAL_STRIDE = max(1, round(EXTRACTED_FPS / PRETRAINED_EVAL_FPS))
TEMPORAL_ABLATION_STRIDE = 3

VAL_CLIPS = 10
TEST_CLIPS = 10
FLIP_TTA = True
EVAL_BATCH_SIZE = 1
NUM_WORKERS = 2

# -----------------------------
# Optimizer and regularization
# -----------------------------
N_EPOCHS = 45
MINIMUM_EPOCHS = 15
WARMUP_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 12
MINIMUM_LR_FACTOR = 0.01
UNFREEZE_EPOCH = 2

DROPOUT = 0.35
LEARNING_RATE = 3e-4
BACKBONE_LR_MULTIPLIER = 0.10
LAYER_DECAY = 0.90
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.10
FREEZE_BATCH_NORM = False
USE_AMP = True

# Clip-consistent augmentation.
TRAIN_CROP_SCALE = (0.60, 1.00)
TRAIN_CROP_RATIO = (0.75, 1.333)
COLOR_JITTER_BRIGHTNESS = 0.20
COLOR_JITTER_CONTRAST = 0.20
COLOR_JITTER_SATURATION = 0.20
COLOR_JITTER_HUE = 0.05
TRAIN_HORIZONTAL_FLIP_PROBABILITY = 0.50
RANDOM_ERASING_PROBABILITY = 0.10
RANDOM_ERASING_SCALE = (0.02, 0.20)
RANDOM_ERASING_RATIO = (0.30, 3.30)

# -----------------------------
# Weights & Biases
# -----------------------------
WANDB_PROJECT = 'hmdb51-videomaev2-v7'
WANDB_ENTITY = ''
WANDB_MODE = 'online'  # online, offline, or disabled

# -----------------------------
# Workflow controls
# -----------------------------
RUN_SMOKE_TEST = True
RUN_FULL_TRAINING = True
RUN_MULTI_SEED_VALIDATION = False
RUN_TEMPORAL_STRIDE_ABLATION = False
RUN_EVALUATION = False  # Keep False while tuning.
RUN_PYLINT = False

print('Repository ZIP:', REPO_ZIP)
print('Dataset ZIP:', DATASET_ZIP)
print('Primary results:', RESULTS_DIR)
print('Variant / architecture:', VIDEOMAE_VARIANT, ARCHITECTURE)
print('Head LR:', LEARNING_RATE)
print('Top backbone LR:', LEARNING_RATE * BACKBONE_LR_MULTIPLIER)
print('Layer decay:', LAYER_DECAY)
print('Mini-batch / accumulation:', BATCH_SIZE, GRADIENT_ACCUMULATION_STEPS)
print('Effective batch:', BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print('Training seed / fixed split seed:', SEED, SPLIT_SEED)
print('Extracted FPS assumption:', EXTRACTED_FPS)
print('Temporal stride / sampled FPS:', TEMPORAL_STRIDE, EXTRACTED_FPS / TEMPORAL_STRIDE)
SOURCE_FRAME_SPAN = 1 + (FRAMES_PER_VIDEO - 1) * TEMPORAL_STRIDE
print('Source frames spanned:', SOURCE_FRAME_SPAN)
print('Approximate clip duration:', SOURCE_FRAME_SPAN / EXTRACTED_FPS, 'seconds')
print('Validation/test clips:', VAL_CLIPS, TEST_CLIPS)
print('Evaluation clip-forward chunk:', EVAL_CLIP_CHUNK_SIZE)
print('Reference MViT split:', REFERENCE_MVIT_SPLIT)
print('Test evaluation enabled:', RUN_EVALUATION)


Repository ZIP: /content/drive/MyDrive/Homework_4/hmdb51_repo_videomaev2_distilled_v7.zip
Dataset ZIP: /content/drive/MyDrive/Homework_4/HMDB51.zip
Primary results: /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42
Variant / architecture: small videomaev2_vit_s_distilled
Head LR: 0.0003
Top backbone LR: 2.9999999999999997e-05
Layer decay: 0.9
Mini-batch / accumulation: 2 4
Effective batch: 8
Training seed / fixed split seed: 42 42
Extracted FPS assumption: 30.0
Temporal stride / sampled FPS: 4 7.5
Source frames spanned: 61
Approximate clip duration: 2.033333333333333 seconds
Validation/test clips: 10 10
Evaluation clip-forward chunk: 1
Reference MViT split: /content/drive/MyDrive/Homework_4/results_mvit_generalization_v6_seed42/splits.json
Test evaluation enabled: False


## 3. Validate configuration and prepare all official split files


In [3]:
from pathlib import Path
import hashlib
import os
import shutil
import subprocess
import sys

required_names = [
    'DRIVE_ROOT', 'PROJECT_ROOT', 'RESULTS_DIR', 'MULTI_SEED_ROOT',
    'REPO_ZIP', 'REPO_EXTRACT_ROOT', 'DATASET_ZIP',
    'DATASET_EXTRACT_ROOT', 'SPLIT_PROTOCOL', 'OFFICIAL_SPLIT_DIR',
    'OFFICIAL_SPLIT_NUMBER', 'OFFICIAL_VALIDATION_SIZE',
    'FINGERPRINT_SPLIT_AUDIT', 'OFFICIAL_DUPLICATE_POLICY',
    'SPLIT_SEED', 'SEED', 'MULTI_SEED_VALUES',
    'AUTO_DOWNLOAD_OFFICIAL_SPLITS', 'OFFICIAL_SPLIT_HF_REPO',
    'OFFICIAL_SPLIT_HF_REVISION', 'OFFICIAL_SPLIT_ARCHIVE_NAME',
    'OFFICIAL_SPLIT_ARCHIVE', 'OFFICIAL_SPLIT_ARCHIVE_MD5',
    'OFFICIAL_SPLIT_ARCHIVE_SHA256', 'OFFICIAL_SPLIT_EXTRACT_ROOT',
    'WANDB_PROJECT', 'WANDB_MODE', 'VIDEOMAE_VARIANT', 'ARCHITECTURE',
    'N_CLASSES', 'N_EPOCHS', 'MINIMUM_EPOCHS', 'FRAMES_PER_VIDEO',
    'EXTRACTED_FPS', 'PRETRAINED_EVAL_FPS', 'TEMPORAL_STRIDE',
    'TEMPORAL_ABLATION_STRIDE', 'IMAGE_SIZE', 'VAL_CLIPS', 'TEST_CLIPS',
    'FLIP_TTA', 'BATCH_SIZE', 'GRADIENT_ACCUMULATION_STEPS',
    'EVAL_BATCH_SIZE', 'EVAL_CLIP_CHUNK_SIZE', 'NUM_WORKERS',
    'DROPOUT', 'LEARNING_RATE', 'BACKBONE_LR_MULTIPLIER',
    'WEIGHT_DECAY', 'LABEL_SMOOTHING', 'WARMUP_EPOCHS',
    'MINIMUM_LR_FACTOR', 'UNFREEZE_EPOCH', 'EARLY_STOPPING_PATIENCE',
    'FREEZE_BATCH_NORM', 'USE_AMP', 'DROP_PATH_RATE', 'LAYER_DECAY',
    'HEAD_INIT_SCALE', 'GRADIENT_CHECKPOINTING',
    'VERIFY_PRETRAINED_SHA256', 'PRETRAINED_CACHE_DIR',
    'PREFETCH_PRETRAINED_WEIGHTS', 'RUN_CHECKPOINT_PREFLIGHT',
    'REFERENCE_MVIT_SPLIT', 'REUSE_MVIT_SPLIT_IF_AVAILABLE',
    'TRAIN_CROP_SCALE', 'TRAIN_CROP_RATIO',
    'COLOR_JITTER_BRIGHTNESS', 'COLOR_JITTER_CONTRAST',
    'COLOR_JITTER_SATURATION', 'COLOR_JITTER_HUE',
    'TRAIN_HORIZONTAL_FLIP_PROBABILITY',
    'RANDOM_ERASING_PROBABILITY', 'RANDOM_ERASING_SCALE',
    'RANDOM_ERASING_RATIO', 'RUN_MULTI_SEED_VALIDATION',
    'RUN_TEMPORAL_STRIDE_ABLATION',
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise NameError(f'Missing configuration variables: {missing}')

if VIDEOMAE_VARIANT not in {'small', 'base'}:
    raise ValueError("VIDEOMAE_VARIANT must be 'small' or 'base'")
expected_architecture = (
    'videomaev2_vit_s_distilled'
    if VIDEOMAE_VARIANT == 'small'
    else 'videomaev2_vit_b_distilled'
)
if ARCHITECTURE != expected_architecture:
    raise ValueError(
        f'ARCHITECTURE={ARCHITECTURE!r} is inconsistent with '
        f'VIDEOMAE_VARIANT={VIDEOMAE_VARIANT!r}'
    )
if FRAMES_PER_VIDEO != 16 or IMAGE_SIZE != 224:
    raise ValueError('The included VideoMAE V2 checkpoints require 16 frames at 224x224')
if BACKBONE_LR_MULTIPLIER <= 0 or LEARNING_RATE <= 0:
    raise ValueError('Learning rates and multipliers must be positive')
if GRADIENT_ACCUMULATION_STEPS <= 0 or BATCH_SIZE <= 0:
    raise ValueError('Batch size and accumulation steps must be positive')
if EVAL_BATCH_SIZE <= 0 or EVAL_CLIP_CHUNK_SIZE <= 0:
    raise ValueError('Evaluation batch and clip chunk sizes must be positive')
if EXTRACTED_FPS <= 0 or PRETRAINED_EVAL_FPS <= 0:
    raise ValueError('FPS values must be positive')
if not 0 <= LABEL_SMOOTHING < 1:
    raise ValueError('LABEL_SMOOTHING must be in [0, 1)')
if not 0 < MINIMUM_LR_FACTOR <= 1:
    raise ValueError('MINIMUM_LR_FACTOR must be in (0, 1]')
if not 0 < LAYER_DECAY <= 1:
    raise ValueError('LAYER_DECAY must be in (0, 1]')
if not 0 <= DROP_PATH_RATE < 1:
    raise ValueError('DROP_PATH_RATE must be in [0, 1)')
if HEAD_INIT_SCALE <= 0:
    raise ValueError('HEAD_INIT_SCALE must be positive')
if UNFREEZE_EPOCH < 0 or EARLY_STOPPING_PATIENCE < 0 or MINIMUM_EPOCHS < 0:
    raise ValueError('Epoch settings must be zero or greater')
if SPLIT_PROTOCOL not in {'random', 'official'}:
    raise ValueError("SPLIT_PROTOCOL must be 'random' or 'official'")
if OFFICIAL_SPLIT_NUMBER not in {1, 2, 3}:
    raise ValueError('OFFICIAL_SPLIT_NUMBER must be 1, 2, or 3')
if OFFICIAL_DUPLICATE_POLICY not in {'error', 'drop_train', 'allow'}:
    raise ValueError(
        "OFFICIAL_DUPLICATE_POLICY must be 'error', 'drop_train', or 'allow'"
    )
if not 0 < TRAIN_CROP_SCALE[0] <= TRAIN_CROP_SCALE[1] <= 1:
    raise ValueError('TRAIN_CROP_SCALE must satisfy 0 < min <= max <= 1')
if not 0 < TRAIN_CROP_RATIO[0] <= TRAIN_CROP_RATIO[1]:
    raise ValueError('TRAIN_CROP_RATIO must satisfy 0 < min <= max')
for name, value in {
    'COLOR_JITTER_BRIGHTNESS': COLOR_JITTER_BRIGHTNESS,
    'COLOR_JITTER_CONTRAST': COLOR_JITTER_CONTRAST,
    'COLOR_JITTER_SATURATION': COLOR_JITTER_SATURATION,
}.items():
    if value < 0:
        raise ValueError(f'{name} must be zero or greater')
if not 0 <= COLOR_JITTER_HUE <= 0.5:
    raise ValueError('COLOR_JITTER_HUE must be in [0, 0.5]')
if not 0 <= TRAIN_HORIZONTAL_FLIP_PROBABILITY <= 1:
    raise ValueError('TRAIN_HORIZONTAL_FLIP_PROBABILITY must be in [0, 1]')
if not 0 <= RANDOM_ERASING_PROBABILITY <= 1:
    raise ValueError('RANDOM_ERASING_PROBABILITY must be in [0, 1]')
if not 0 < RANDOM_ERASING_SCALE[0] <= RANDOM_ERASING_SCALE[1] <= 1:
    raise ValueError('RANDOM_ERASING_SCALE must satisfy 0 < min <= max <= 1')
if not 0 < RANDOM_ERASING_RATIO[0] <= RANDOM_ERASING_RATIO[1]:
    raise ValueError('RANDOM_ERASING_RATIO must satisfy 0 < min <= max')
if not MULTI_SEED_VALUES:
    raise ValueError('MULTI_SEED_VALUES cannot be empty')
if TEMPORAL_ABLATION_STRIDE <= 0:
    raise ValueError('TEMPORAL_ABLATION_STRIDE must be positive')

HMDB51_CLASS_NAMES = (
    'brush_hair', 'cartwheel', 'catch', 'chew', 'clap', 'climb',
    'climb_stairs', 'dive', 'draw_sword', 'dribble', 'drink', 'eat',
    'fall_floor', 'fencing', 'flic_flac', 'golf', 'handstand', 'hit',
    'hug', 'jump', 'kick', 'kick_ball', 'kiss', 'laugh', 'pick', 'pour',
    'pullup', 'punch', 'push', 'pushup', 'ride_bike', 'ride_horse', 'run',
    'shake_hands', 'shoot_ball', 'shoot_bow', 'shoot_gun', 'sit', 'situp',
    'smile', 'smoke', 'somersault', 'stand', 'swing_baseball', 'sword',
    'sword_exercise', 'talk', 'throw', 'turn', 'walk', 'wave',
)
if len(HMDB51_CLASS_NAMES) != N_CLASSES:
    raise RuntimeError('HMDB51 class-name configuration is inconsistent')


def _digest(path: Path, algorithm: str) -> str:
    hasher = hashlib.new(algorithm)
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            hasher.update(chunk)
    return hasher.hexdigest()


def _expected_names(split_number: int):
    return {
        f'{class_name}_test_split{split_number}.txt'
        for class_name in HMDB51_CLASS_NAMES
    }


def _validate_split_file(path: Path):
    counts = {0: 0, 1: 0, 2: 0}
    rows = path.read_text(encoding='utf-8-sig').splitlines()
    if not rows:
        raise RuntimeError(f'Official split file is empty: {path}')
    for line_number, raw_line in enumerate(rows, start=1):
        line = raw_line.strip()
        if not line:
            continue
        parts = line.rsplit(maxsplit=1)
        if len(parts) != 2 or parts[1] not in {'0', '1', '2'}:
            raise RuntimeError(
                f'Invalid row in {path.name} at line {line_number}: {raw_line!r}'
            )
        video_name, tag_text = parts
        if not video_name.lower().endswith('.avi'):
            raise RuntimeError(
                f'Expected an AVI filename in {path.name} at line {line_number}'
            )
        counts[int(tag_text)] += 1
    if counts[1] != 70 or counts[2] != 30:
        raise RuntimeError(
            f'{path.name} has train/test counts {counts[1]}/{counts[2]}; '
            'expected the official 70/30 protocol.'
        )


def _validate_complete_directory(directory: Path, require_all_splits: bool = True):
    directory = Path(directory)
    splits = (1, 2, 3) if require_all_splits else (OFFICIAL_SPLIT_NUMBER,)
    for split_number in splits:
        expected = _expected_names(split_number)
        present = {
            path.name for path in directory.glob(f'*_test_split{split_number}.txt')
        }
        if present != expected:
            missing_names = sorted(expected - present)
            extra_names = sorted(present - expected)
            raise RuntimeError(
                f'Incomplete official split {split_number} in {directory}: '
                f'found {len(present)} of {len(expected)} expected files. '
                f'Missing examples: {missing_names[:5]}; '
                f'extra examples: {extra_names[:5]}'
            )
        for file_name in sorted(expected):
            _validate_split_file(directory / file_name)
    return directory.resolve()


def _find_complete_directories(search_root: Path):
    search_root = Path(search_root)
    if not search_root.is_dir():
        return []
    candidates = set()
    anchor = f'brush_hair_test_split{OFFICIAL_SPLIT_NUMBER}.txt'
    if (search_root / anchor).is_file():
        candidates.add(search_root.resolve())
    candidates.update(path.parent.resolve() for path in search_root.rglob(anchor))
    complete = []
    for candidate in candidates:
        try:
            _validate_complete_directory(candidate, require_all_splits=True)
        except RuntimeError:
            continue
        complete.append(candidate)
    return sorted(set(complete), key=lambda path: (len(path.parts), str(path)))


def _ensure_huggingface_hub():
    try:
        import huggingface_hub  # noqa: F401
    except ImportError:
        print('Installing huggingface_hub for the official split download...')
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'],
            check=True,
        )


def _download_verified_archive() -> Path:
    archive = Path(OFFICIAL_SPLIT_ARCHIVE)
    if archive.is_file():
        md5 = _digest(archive, 'md5')
        sha256 = _digest(archive, 'sha256')
        if (
            md5.lower() == OFFICIAL_SPLIT_ARCHIVE_MD5.lower()
            and sha256.lower() == OFFICIAL_SPLIT_ARCHIVE_SHA256.lower()
        ):
            print('Using verified official split archive:', archive)
            return archive
        print('Removing an invalid cached split archive:', archive)
        archive.unlink()

    _ensure_huggingface_hub()
    from huggingface_hub import hf_hub_download

    print(
        'Downloading the official HMDB51 split archive from '
        f'{OFFICIAL_SPLIT_HF_REPO}...'
    )
    downloaded = Path(
        hf_hub_download(
            repo_id=OFFICIAL_SPLIT_HF_REPO,
            repo_type='dataset',
            filename=OFFICIAL_SPLIT_ARCHIVE_NAME,
            revision=OFFICIAL_SPLIT_HF_REVISION,
        )
    )
    archive.parent.mkdir(parents=True, exist_ok=True)
    if downloaded.resolve() != archive.resolve():
        shutil.copy2(downloaded, archive)

    md5 = _digest(archive, 'md5')
    sha256 = _digest(archive, 'sha256')
    if md5.lower() != OFFICIAL_SPLIT_ARCHIVE_MD5.lower():
        archive.unlink(missing_ok=True)
        raise RuntimeError(
            f'Official archive MD5 mismatch: expected '
            f'{OFFICIAL_SPLIT_ARCHIVE_MD5}, found {md5}'
        )
    if sha256.lower() != OFFICIAL_SPLIT_ARCHIVE_SHA256.lower():
        archive.unlink(missing_ok=True)
        raise RuntimeError(
            f'Official archive SHA-256 mismatch: expected '
            f'{OFFICIAL_SPLIT_ARCHIVE_SHA256}, found {sha256}'
        )
    print('Official split archive checksums verified.')
    return archive


def _ensure_rar_extractor():
    for command in ('unar', 'unrar', '7z'):
        executable = shutil.which(command)
        if executable:
            return command, executable
    print("Installing 'unar' so Colab can extract the official RAR archive...")
    environment = os.environ.copy()
    environment['DEBIAN_FRONTEND'] = 'noninteractive'
    subprocess.run(['apt-get', 'update', '-qq'], check=True, env=environment)
    subprocess.run(
        ['apt-get', 'install', '-y', '-qq', 'unar'],
        check=True,
        env=environment,
    )
    executable = shutil.which('unar')
    if executable is None:
        raise RuntimeError("The 'unar' installation completed, but unar was not found")
    return 'unar', executable


def _extract_archive(archive: Path, output_directory: Path):
    kind, executable = _ensure_rar_extractor()
    output_directory.mkdir(parents=True, exist_ok=True)
    if kind == 'unar':
        command = [
            executable, '-q', '-f', '-o', str(output_directory), str(archive)
        ]
    elif kind == 'unrar':
        command = [
            executable, 'x', '-o+', '-y', str(archive), str(output_directory) + os.sep
        ]
    else:
        command = [
            executable, 'x', '-y', f'-o{output_directory}', str(archive)
        ]
    print('Extracting official split archive...')
    subprocess.run(command, check=True)


def _prepare_official_splits() -> Path:
    configured = Path(OFFICIAL_SPLIT_DIR).expanduser()

    existing = _find_complete_directories(configured)
    if not existing:
        existing = _find_complete_directories(PROJECT_ROOT)
    if len(existing) == 1:
        resolved = _validate_complete_directory(
            existing[0], require_all_splits=True
        )
        print('Resolved existing official split directory:', resolved)
        return resolved
    if len(existing) > 1:
        formatted = '\n  - '.join(str(path) for path in existing[:10])
        raise RuntimeError(
            'Multiple complete official split directories were found. Set '
            'OFFICIAL_SPLIT_DIR explicitly:\n  - ' + formatted
        )

    if not AUTO_DOWNLOAD_OFFICIAL_SPLITS:
        raise FileNotFoundError(
            'No complete official split directory was found and automatic '
            'download is disabled.'
        )

    archive = _download_verified_archive()
    extraction_root = Path(OFFICIAL_SPLIT_EXTRACT_ROOT)
    if extraction_root.exists():
        shutil.rmtree(extraction_root)
    extraction_root.mkdir(parents=True, exist_ok=True)
    _extract_archive(archive, extraction_root)

    extracted = _find_complete_directories(extraction_root)
    if len(extracted) != 1:
        raise RuntimeError(
            'The verified official archive was extracted, but exactly one '
            f'complete split directory was expected; found {extracted}'
        )
    source = _validate_complete_directory(extracted[0], require_all_splits=True)

    # Replace only split text files, preserving unrelated project files.
    configured.mkdir(parents=True, exist_ok=True)
    for old_file in configured.glob('*_test_split*.txt'):
        old_file.unlink()
    for source_file in sorted(source.glob('*_test_split*.txt')):
        shutil.copy2(source_file, configured / source_file.name)

    resolved = _validate_complete_directory(configured, require_all_splits=True)
    print('Prepared official split directory:', resolved)
    print('Total official split files:', len(list(resolved.glob('*_test_split*.txt'))))
    return resolved


if SPLIT_PROTOCOL == 'official':
    OFFICIAL_SPLIT_DIR = _prepare_official_splits()
    print('Selected official split:', OFFICIAL_SPLIT_NUMBER)
    print(
        'Selected split class files:',
        len(list(OFFICIAL_SPLIT_DIR.glob(
            f'*_test_split{OFFICIAL_SPLIT_NUMBER}.txt'
        ))),
    )

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
write_test = RESULTS_DIR / '.write_test'
write_test.write_text('ok', encoding='utf-8')
write_test.unlink()
print('Configuration and Drive write access verified.')


Resolved existing official split directory: /content/drive/MyDrive/Homework_4/testTrainMulti_7030_splits
Selected official split: 1
Selected split class files: 51
Configuration and Drive write access verified.


## 4. Verify the GPU


In [4]:
import subprocess
import torch

subprocess.run(['nvidia-smi'], check=False)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. Select a GPU runtime in Colab.')

properties = torch.cuda.get_device_properties(0)
total_gib = properties.total_memory / (1024 ** 3)
print('CUDA device:', properties.name)
print(f'GPU memory: {total_gib:.1f} GiB')
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)

if VIDEOMAE_VARIANT == 'base' and total_gib < 20:
    print(
        'Warning: ViT-B is a high-memory run. The notebook already uses '
        'batch=1, accumulation=8, activation checkpointing, and one-clip '
        'evaluation chunks. Start with the smoke test.'
    )


CUDA device: NVIDIA A100-SXM4-40GB
GPU memory: 39.5 GiB
PyTorch: 2.11.0+cu128
CUDA runtime: 12.8


## 5. Extract and locate the v7 repository


In [5]:
import shutil
import zipfile

if not REPO_ZIP.exists():
    raise FileNotFoundError(
        f'Repository ZIP not found: {REPO_ZIP}\n'
        'Upload hmdb51_repo_videomaev2_distilled_v7.zip to Homework_4.'
    )

if REPO_EXTRACT_ROOT.exists():
    shutil.rmtree(REPO_EXTRACT_ROOT)
REPO_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(REPO_ZIP) as archive:
    archive.extractall(REPO_EXTRACT_ROOT)

run_files = list(REPO_EXTRACT_ROOT.rglob('run.py'))
videomae_files = list(REPO_EXTRACT_ROOT.rglob('videomaev2.py'))
if len(run_files) != 1 or len(videomae_files) != 1:
    raise RuntimeError(
        'Expected exactly one run.py and one videomaev2.py; found '
        f'{run_files} and {videomae_files}'
    )
REPO_DIR = run_files[0].parent
if videomae_files[0].parent != REPO_DIR:
    raise RuntimeError('run.py and videomaev2.py were not extracted together')
print('Repository directory:', REPO_DIR)
print('Repository files:', sorted(path.name for path in REPO_DIR.iterdir()))


Repository directory: /content/hmdb51_repo_videomaev2_distilled_v7/hmdb51_repo_videomaev2_distilled_v7
Repository files: ['ACCURACY_UPDATE.md', 'AUDIT_REPORT.md', 'HMDB51_Colab_VideoMAEv2_Distilled_v7.ipynb', 'IMPROVEMENTS.md', 'OFFICIAL_DUPLICATE_POLICY.md', 'README.md', 'SPLIT_AND_DUPLICATE_POLICY.md', 'VIDEOMAEV2_EXPERIMENT_V7.md', 'models.py', 'preflight_videomaev2.py', 'requirements.txt', 'run.py', 'run_training.py', 'test.py', 'test.sh', 'tests', 'train.py', 'train.sh', 'utils.py', 'video_datasets.py', 'videomaev2.py']


## 6. Install dependencies


In [6]:
import sys
import subprocess

requirements = REPO_DIR / 'requirements.txt'
if requirements.exists():
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install', '-q',
            '--upgrade-strategy', 'only-if-needed', '-r', str(requirements)
        ],
        check=True,
    )

import torch
import torchvision
import huggingface_hub

if not hasattr(torch.nn.functional, 'scaled_dot_product_attention'):
    raise RuntimeError('This PyTorch build lacks scaled-dot-product attention')
print('Dependencies installed.')
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('huggingface_hub:', huggingface_hub.__version__)


Dependencies installed.
torch: 2.11.0+cu128
torchvision: 0.26.0+cu128
huggingface_hub: 1.23.0


## 7. Authenticate optionally and verify the VideoMAE V2 checkpoint


In [7]:
import json
import os
import sys

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN loaded from Colab secrets.')
else:
    print('No HF_TOKEN secret found. The public checkpoint can still be downloaded.')

from videomaev2 import (
    checkpoint_spec_for_architecture,
    download_verified_checkpoint,
    preflight_videomaev2_checkpoint,
    verify_checkpoint_file,
)

PRETRAINED_CACHE_DIR.mkdir(parents=True, exist_ok=True)
specification = checkpoint_spec_for_architecture(ARCHITECTURE)
print('Pinned checkpoint specification:')
print(json.dumps(specification.to_dict(), indent=2))

RESOLVED_PRETRAINED_CHECKPOINT = None
if LOCAL_PRETRAINED_CHECKPOINT is not None:
    RESOLVED_PRETRAINED_CHECKPOINT = Path(LOCAL_PRETRAINED_CHECKPOINT)
    if VERIFY_PRETRAINED_SHA256:
        verify_checkpoint_file(RESOLVED_PRETRAINED_CHECKPOINT, specification)
elif PREFETCH_PRETRAINED_WEIGHTS:
    RESOLVED_PRETRAINED_CHECKPOINT = download_verified_checkpoint(
        ARCHITECTURE,
        cache_dir=PRETRAINED_CACHE_DIR,
        verify_sha256=VERIFY_PRETRAINED_SHA256,
    )

if RESOLVED_PRETRAINED_CHECKPOINT is not None:
    print('Resolved pretrained checkpoint:', RESOLVED_PRETRAINED_CHECKPOINT)
    print(
        'Checkpoint size (MiB):',
        RESOLVED_PRETRAINED_CHECKPOINT.stat().st_size / (1024 ** 2),
    )

if RUN_CHECKPOINT_PREFLIGHT:
    preflight_report = preflight_videomaev2_checkpoint(
        ARCHITECTURE,
        cache_dir=PRETRAINED_CACHE_DIR,
        verify_sha256=VERIFY_PRETRAINED_SHA256,
        pretrained_checkpoint=RESOLVED_PRETRAINED_CHECKPOINT,
    )
    preflight_path = RESULTS_DIR / 'pretrained_preflight.json'
    preflight_path.write_text(
        json.dumps(preflight_report, indent=2, sort_keys=True),
        encoding='utf-8',
    )
    print(json.dumps(preflight_report, indent=2, sort_keys=True))
    print('Preflight report:', preflight_path)
    if preflight_report['matched_backbone_parameter_fraction'] < 0.97:
        raise RuntimeError('Checkpoint preflight did not meet the 97% match threshold')
else:
    print('Checkpoint structural preflight skipped.')


No HF_TOKEN secret found. The public checkpoint can still be downloaded.
Pinned checkpoint specification:
{
  "architecture": "videomaev2_vit_s_distilled",
  "variant": "small",
  "repo_id": "OpenGVLab/VideoMAE2",
  "revision": "706cc172d65ebd4dedbee3f9c0183a93df9fa125",
  "filename": "distill/vit_s_k710_dl_from_giant.pth",
  "sha256": "24fb71687fa3671b8387cadfbcbab0f72af695692e93cf1ecc82caa888626172",
  "file_size": 44334609,
  "pretraining_classes": 710,
  "embed_dim": 384,
  "depth": 12,
  "num_heads": 6
}


distill/vit_s_k710_dl_from_giant.pth: reconstructing file:   0%|          |  0.00B / 44.3MB            

distill/vit_s_k710_dl_from_giant.pth: downloading bytes:           |  0.00B            

Resolved pretrained checkpoint: /content/drive/MyDrive/Homework_4/videomaev2_pretrained_cache/models--OpenGVLab--VideoMAE2/snapshots/706cc172d65ebd4dedbee3f9c0183a93df9fa125/distill/vit_s_k710_dl_from_giant.pth
Checkpoint size (MiB): 42.28077793121338
Loaded official VideoMAE V2 distilled weights: videomaev2_vit_s_distilled, backbone match=100.00%, matched tensors=162
{
  "architecture": "videomaev2_vit_s_distilled",
  "backbone_parameter_count": 21879936,
  "checkpoint_tensor_count": 162,
  "collisions": [],
  "feature_size": 384,
  "matched_backbone_parameter_fraction": 1.0,
  "matched_tensor_count": 162,
  "missing_after_load": [],
  "missing_backbone_keys": [],
  "resolved_checkpoint_path": "/content/drive/MyDrive/Homework_4/videomaev2_pretrained_cache/models--OpenGVLab--VideoMAE2/blobs/24fb71687fa3671b8387cadfbcbab0f72af695692e93cf1ecc82caa888626172",
  "unexpected_after_load": []
}
Preflight report: /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42/pretrained_p

## 8. Extract and locate the HMDB51 frame dataset


In [8]:
if not DATASET_ZIP.exists():
    raise FileNotFoundError(f'Dataset ZIP not found: {DATASET_ZIP}')

if DATASET_EXTRACT_ROOT.exists():
    shutil.rmtree(DATASET_EXTRACT_ROOT)
DATASET_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Extracting {DATASET_ZIP} to {DATASET_EXTRACT_ROOT} ...')
shutil.unpack_archive(str(DATASET_ZIP), str(DATASET_EXTRACT_ROOT))
print('Dataset extraction complete.')

# The correct frame root has exactly 51 immediate class directories.
def class_directory_count(path: Path) -> int:
    if not path.is_dir():
        return 0
    return sum(child.is_dir() for child in path.iterdir())

candidates = [DATASET_EXTRACT_ROOT]
candidates.extend(path for path in DATASET_EXTRACT_ROOT.rglob('*') if path.is_dir())
matching = [path for path in candidates if class_directory_count(path) == N_CLASSES]

if not matching:
    raise FileNotFoundError(
        f'Could not find a directory containing exactly {N_CLASSES} class folders '
        f'under {DATASET_EXTRACT_ROOT}'
    )

# Prefer the shallowest matching directory.
DATASET_DIR = min(matching, key=lambda path: len(path.parts))
print('Detected frame dataset directory:', DATASET_DIR)


Extracting /content/drive/MyDrive/Homework_4/HMDB51.zip to /content/hmdb51_data ...
Dataset extraction complete.
Detected frame dataset directory: /content/hmdb51_data/HMDB51


## 9. Validate the dataset structure and temporal coverage


In [9]:
classes = sorted(path for path in DATASET_DIR.iterdir() if path.is_dir())
if len(classes) != N_CLASSES:
    raise RuntimeError(f'Expected {N_CLASSES} classes, found {len(classes)}')

SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
video_count = 0
empty_video_dirs = []
frame_counts = []
for class_dir in classes:
    for video_dir in (path for path in class_dir.iterdir() if path.is_dir()):
        video_count += 1
        frames = [
            path for path in video_dir.iterdir()
            if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
        ]
        frame_counts.append(len(frames))
        if not frames:
            empty_video_dirs.append(str(video_dir))

required_span = 1 + (FRAMES_PER_VIDEO - 1) * TEMPORAL_STRIDE
uniform_sampling_count = sum(count < required_span for count in frame_counts)
fewer_than_output_count = sum(count < FRAMES_PER_VIDEO for count in frame_counts)

print('Classes:', len(classes))
print('First 10 classes:', [path.name for path in classes[:10]])
print('Video directories:', video_count)
print('Frame-count range:', (min(frame_counts), max(frame_counts)) if frame_counts else None)
print('Requested temporal span:', required_span)
print(
    'Videos using uniform full-duration sampling:',
    f'{uniform_sampling_count}/{video_count}',
    f'({uniform_sampling_count / video_count:.1%})' if video_count else None,
)
print('Videos with fewer frames than output clip:', fewer_than_output_count)
print('Empty video directories:', len(empty_video_dirs))
if empty_video_dirs:
    print('First empty directories:', empty_video_dirs[:10])


Classes: 51
First 10 classes: ['brush_hair', 'cartwheel', 'catch', 'chew', 'clap', 'climb', 'climb_stairs', 'dive', 'draw_sword', 'dribble']
Video directories: 6766
Frame-count range: (4, 213)
Requested temporal span: 61
Videos using uniform full-duration sampling: 6615/6766 (97.8%)
Videos with fewer frames than output clip: 2498
Empty video directories: 0


## 10. Log in to Weights & Biases


In [10]:
import os

if WANDB_MODE == 'online':
    import wandb
    try:
        from google.colab import userdata
        wandb_key = userdata.get('WANDB_API_KEY')
    except Exception:
        wandb_key = None

    if wandb_key:
        os.environ['WANDB_API_KEY'] = wandb_key
        login_ok = wandb.login(key=wandb_key, relogin=True, verify=True)
    else:
        login_ok = wandb.login(relogin=True, verify=True)

    if not login_ok:
        raise RuntimeError('W&B login did not succeed.')
    print('W&B online login verified.')
elif WANDB_MODE == 'offline':
    os.environ['WANDB_MODE'] = 'offline'
    print('W&B offline mode enabled.')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B disabled.')


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tim-vaught12 (tim-vaught12-johns-hopkins-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B online login verified.


## 11. Command, fixed-split, and logging helpers


In [11]:
import os
import shlex
import shutil
import subprocess
import sys

os.chdir(REPO_DIR)


def run_and_log(command, filename, log_directory=RESULTS_DIR):
    log_directory.mkdir(parents=True, exist_ok=True)
    log_path = log_directory / filename
    print('Running:', ' '.join(shlex.quote(str(item)) for item in command))
    with log_path.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            [str(item) for item in command],
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return log_path


def prepare_reused_split(output_dir: Path):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    destination = output_dir / 'splits.json'
    if destination.is_file():
        print('Using existing split file:', destination)
        return destination

    candidates = []
    primary_split = RESULTS_DIR / 'splits.json'
    if output_dir.resolve() != RESULTS_DIR.resolve():
        candidates.append(primary_split)
    if REUSE_MVIT_SPLIT_IF_AVAILABLE:
        candidates.append(REFERENCE_MVIT_SPLIT)

    for source in candidates:
        source = Path(source)
        if source.is_file():
            shutil.copy2(source, destination)
            print('Copied fixed split:', source, '->', destination)
            return destination
    return None


def train_command(
    epochs,
    batch_size,
    project,
    output_dir,
    *,
    training_seed=SEED,
    split_seed=SPLIT_SEED,
    temporal_stride=TEMPORAL_STRIDE,
    validation_clips=VAL_CLIPS,
    test_clips=TEST_CLIPS,
    minimum_epochs=MINIMUM_EPOCHS,
    unfreeze_epoch=UNFREEZE_EPOCH,
):
    output_dir = Path(output_dir)
    command = [
        sys.executable, 'run.py',
        '--frame_dir', str(DATASET_DIR),
        '--mode', 'train',
        '--output_dir', str(output_dir),
        '--split_protocol', SPLIT_PROTOCOL,
        '--architecture', ARCHITECTURE,
        '--n_classes', str(N_CLASSES),
        '--fr_per_vid', str(FRAMES_PER_VIDEO),
        '--image_size', str(IMAGE_SIZE),
        '--temporal_stride', str(temporal_stride),
        '--val_clips', str(validation_clips),
        '--test_clips', str(test_clips),
        '--flip_tta', str(FLIP_TTA).lower(),
        '--batch_size', str(batch_size),
        '--gradient_accumulation_steps', str(GRADIENT_ACCUMULATION_STEPS),
        '--eval_batch_size', str(EVAL_BATCH_SIZE),
        '--eval_clip_chunk_size', str(EVAL_CLIP_CHUNK_SIZE),
        '--workers', str(NUM_WORKERS),
        '--pretrained', 'true',
        '--pretrained_cache_dir', str(PRETRAINED_CACHE_DIR),
        '--verify_pretrained_sha256', str(VERIFY_PRETRAINED_SHA256).lower(),
        '--gradient_checkpointing', str(GRADIENT_CHECKPOINTING).lower(),
        '--drop_path_rate', str(DROP_PATH_RATE),
        '--layer_decay', str(LAYER_DECAY),
        '--head_init_scale', str(HEAD_INIT_SCALE),
        '--freeze_batch_norm', str(FREEZE_BATCH_NORM).lower(),
        '--dropout', str(DROPOUT),
        '--train_crop_scale_min', str(TRAIN_CROP_SCALE[0]),
        '--train_crop_scale_max', str(TRAIN_CROP_SCALE[1]),
        '--train_crop_ratio_min', str(TRAIN_CROP_RATIO[0]),
        '--train_crop_ratio_max', str(TRAIN_CROP_RATIO[1]),
        '--color_jitter_brightness', str(COLOR_JITTER_BRIGHTNESS),
        '--color_jitter_contrast', str(COLOR_JITTER_CONTRAST),
        '--color_jitter_saturation', str(COLOR_JITTER_SATURATION),
        '--color_jitter_hue', str(COLOR_JITTER_HUE),
        '--train_horizontal_flip_probability',
        str(TRAIN_HORIZONTAL_FLIP_PROBABILITY),
        '--random_erasing_probability', str(RANDOM_ERASING_PROBABILITY),
        '--random_erasing_scale_min', str(RANDOM_ERASING_SCALE[0]),
        '--random_erasing_scale_max', str(RANDOM_ERASING_SCALE[1]),
        '--random_erasing_ratio_min', str(RANDOM_ERASING_RATIO[0]),
        '--random_erasing_ratio_max', str(RANDOM_ERASING_RATIO[1]),
        '--learning_rate', str(LEARNING_RATE),
        '--backbone_lr_multiplier', str(BACKBONE_LR_MULTIPLIER),
        '--weight_decay', str(WEIGHT_DECAY),
        '--label_smoothing', str(LABEL_SMOOTHING),
        '--n_epochs', str(epochs),
        '--minimum_epochs', str(min(minimum_epochs, epochs)),
        '--warmup_epochs', str(min(WARMUP_EPOCHS, max(epochs - 1, 0))),
        '--minimum_lr_factor', str(MINIMUM_LR_FACTOR),
        '--unfreeze_epoch', str(unfreeze_epoch),
        '--early_stopping_patience', str(EARLY_STOPPING_PATIENCE),
        '--amp', str(USE_AMP).lower(),
        '--fingerprint_split_audit', str(FINGERPRINT_SPLIT_AUDIT).lower(),
        '--split_seed', str(split_seed),
        '--seed', str(training_seed),
        '--wandb_project', project,
        '--wandb_mode', WANDB_MODE,
    ]

    if RESOLVED_PRETRAINED_CHECKPOINT is not None:
        command.extend([
            '--pretrained_checkpoint', str(RESOLVED_PRETRAINED_CHECKPOINT)
        ])

    if SPLIT_PROTOCOL == 'official':
        if not OFFICIAL_SPLIT_DIR.is_dir():
            raise FileNotFoundError(
                f'Official split directory not found: {OFFICIAL_SPLIT_DIR}'
            )
        command.extend([
            '--official_split_dir', str(OFFICIAL_SPLIT_DIR),
            '--official_split_number', str(OFFICIAL_SPLIT_NUMBER),
            '--official_duplicate_policy', OFFICIAL_DUPLICATE_POLICY,
            '--validation_size', str(OFFICIAL_VALIDATION_SIZE),
        ])

    reused_split = prepare_reused_split(output_dir)
    if reused_split is not None:
        command.extend([
            '--split_file', str(reused_split),
            '--reuse_existing_split', 'true',
        ])

    if WANDB_ENTITY:
        command.extend(['--wandb_entity', WANDB_ENTITY])
    return command


preview = train_command(
    N_EPOCHS,
    BATCH_SIZE,
    WANDB_PROJECT,
    RESULTS_DIR,
)
print('Full training command:')
print(' '.join(shlex.quote(str(item)) for item in preview))


Copied fixed split: /content/drive/MyDrive/Homework_4/results_mvit_generalization_v6_seed42/splits.json -> /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42/splits.json
Full training command:
/usr/bin/python3 run.py --frame_dir /content/hmdb51_data/HMDB51 --mode train --output_dir /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42 --split_protocol official --architecture videomaev2_vit_s_distilled --n_classes 51 --fr_per_vid 16 --image_size 224 --temporal_stride 4 --val_clips 10 --test_clips 10 --flip_tta true --batch_size 2 --gradient_accumulation_steps 4 --eval_batch_size 1 --eval_clip_chunk_size 1 --workers 2 --pretrained true --pretrained_cache_dir /content/drive/MyDrive/Homework_4/videomaev2_pretrained_cache --verify_pretrained_sha256 true --gradient_checkpointing true --drop_path_rate 0.1 --layer_decay 0.9 --head_init_scale 0.001 --freeze_batch_norm false --dropout 0.35 --train_crop_scale_min 0.6 --train_crop_scale_max 1.0 --train_crop_ratio_m

## 12. Artifact helpers


In [12]:
import hashlib
import json

CHECKPOINT_NAME = 'best_model_wts.pt'
SPLITS_NAME = 'splits.json'
HISTORY_NAME = 'training_history.json'
RUN_CONFIG_NAME = 'run_config.json'
DUPLICATE_AUDIT_NAME = 'official_duplicate_audit.json'
PREFLIGHT_NAME = 'pretrained_preflight.json'


def artifact_candidates(filename, destination=RESULTS_DIR):
    return [
        Path(destination) / filename,
        REPO_DIR / filename,
        REPO_DIR / 'models' / filename,
    ]


def find_artifact(filename, destination=RESULTS_DIR):
    return next(
        (path for path in artifact_candidates(filename, destination) if path.exists()),
        None,
    )


def verify_training_artifacts(destination=RESULTS_DIR):
    expected = [CHECKPOINT_NAME, SPLITS_NAME, HISTORY_NAME, RUN_CONFIG_NAME]
    if SPLIT_PROTOCOL == 'official':
        expected.append(DUPLICATE_AUDIT_NAME)
    found = {}
    for filename in expected:
        path = find_artifact(filename, destination)
        if path is None:
            print(f'Artifact not found: {filename}')
        else:
            found[filename] = path
            print(f'Artifact ready: {path}')
    return found


def selected_validation_metrics(history_path):
    history = json.loads(Path(history_path).read_text(encoding='utf-8'))
    accuracies = history['accuracy']['val']
    losses = history['loss']['val']
    if len(accuracies) != len(losses) or not accuracies:
        raise RuntimeError(f'Invalid training history: {history_path}')
    best_index = max(
        range(len(accuracies)),
        key=lambda index: (accuracies[index], -losses[index]),
    )
    return {
        'epoch': best_index + 1,
        'val_accuracy': accuracies[best_index],
        'val_loss': losses[best_index],
        'epochs_trained': len(accuracies),
    }


def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


## 13. One-epoch GPU smoke test


In [13]:
if RUN_SMOKE_TEST:
    smoke_dir = RESULTS_DIR / 'smoke_test_artifacts'
    run_and_log(
        train_command(
            epochs=1,
            batch_size=BATCH_SIZE,
            project=f'{WANDB_PROJECT}-{VIDEOMAE_VARIANT}-smoke',
            output_dir=smoke_dir,
            training_seed=SEED,
            split_seed=SPLIT_SEED,
            temporal_stride=TEMPORAL_STRIDE,
            validation_clips=min(2, VAL_CLIPS),
            test_clips=min(2, TEST_CLIPS),
            minimum_epochs=1,
            unfreeze_epoch=1,
        ),
        'smoke_test_output.txt',
        log_directory=smoke_dir,
    )
    verify_training_artifacts(smoke_dir)

    # When no prior MViT split existed, promote the smoke-test split so the
    # full run and all subsequent runs use exactly the same membership.
    smoke_split = smoke_dir / SPLITS_NAME
    primary_split = RESULTS_DIR / SPLITS_NAME
    if smoke_split.is_file() and not primary_split.is_file():
        shutil.copy2(smoke_split, primary_split)
        print('Promoted smoke-test split to the primary run:', primary_split)
else:
    print('Smoke test skipped.')


Copied fixed split: /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42/splits.json -> /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42/smoke_test_artifacts/splits.json
Running: /usr/bin/python3 run.py --frame_dir /content/hmdb51_data/HMDB51 --mode train --output_dir /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42/smoke_test_artifacts --split_protocol official --architecture videomaev2_vit_s_distilled --n_classes 51 --fr_per_vid 16 --image_size 224 --temporal_stride 4 --val_clips 2 --test_clips 2 --flip_tta true --batch_size 2 --gradient_accumulation_steps 4 --eval_batch_size 1 --eval_clip_chunk_size 1 --workers 2 --pretrained true --pretrained_cache_dir /content/drive/MyDrive/Homework_4/videomaev2_pretrained_cache --verify_pretrained_sha256 true --gradient_checkpointing true --drop_path_rate 0.1 --layer_decay 0.9 --head_init_scale 0.001 --freeze_batch_norm false --dropout 0.35 --train_crop_scale_min 0.6 --train_crop_scale_max 1

## 14. Primary VideoMAE V2 run


In [14]:
if RUN_FULL_TRAINING:
    run_and_log(
        train_command(
            epochs=N_EPOCHS,
            batch_size=BATCH_SIZE,
            project=f'{WANDB_PROJECT}-{VIDEOMAE_VARIANT}',
            output_dir=RESULTS_DIR,
            training_seed=SEED,
            split_seed=SPLIT_SEED,
            temporal_stride=TEMPORAL_STRIDE,
        ),
        'full_training_output.txt',
    )

    saved_artifacts = verify_training_artifacts()
    checkpoint_path = saved_artifacts.get(CHECKPOINT_NAME)
    splits_path = saved_artifacts.get(SPLITS_NAME)
    history_path = saved_artifacts.get(HISTORY_NAME)

    if checkpoint_path is None:
        raise FileNotFoundError(f'Training completed but {CHECKPOINT_NAME} is missing')
    if splits_path is None:
        raise FileNotFoundError(f'Training completed but {SPLITS_NAME} is missing')
    if history_path is not None:
        print('Selected validation metrics:', selected_validation_metrics(history_path))
        print('Split SHA-256:', file_sha256(splits_path))
else:
    print('Primary full training skipped.')


Streaming output truncated to the last 5000 lines.
100%|█████████▉| 1516/1517 [03:48<00:00,  6.97it/s]
                                                   

100%|██████████| 536/536 [03:16<00:00,  3.26it/s]
                                                 
Epoch 031: train loss=0.7440, train acc=99.34%, val loss=1.0577, val acc=89.37%, head lr=7.73e-05, backbone lr=7.73e-06

100%|█████████▉| 1516/1517 [03:45<00:00,  6.97it/s]
                                                   

100%|██████████| 536/536 [03:18<00:00,  3.21it/s]
                                                 
Epoch 032: train loss=0.7423, train acc=99.54%, val loss=1.0556, val acc=89.18%, head lr=6.71e-05, backbone lr=6.71e-06

100%|█████████▉| 1516/1517 [03:47<00:00,  6.91it/s]
                                                   

100%|██████████| 536/536 [03:17<00:00,  3.22it/s]
                                                 
Epoch 033: train loss=0.7381, train acc=99.51%, val loss=1.0545, val acc=89.74%, head lr=5.7

KeyboardInterrupt: 

## 15. Optional fixed-split multi-seed validation


In [15]:
if RUN_MULTI_SEED_VALIDATION:
    MULTI_SEED_ROOT.mkdir(parents=True, exist_ok=True)
    if not (RESULTS_DIR / SPLITS_NAME).is_file():
        raise FileNotFoundError(
            'Run the primary experiment first so all seeds can reuse its split.'
        )
    for training_seed in MULTI_SEED_VALUES:
        seed_output_dir = (
            RESULTS_DIR
            if training_seed == SEED
            else MULTI_SEED_ROOT / f'seed_{training_seed}'
        )
        history_path = seed_output_dir / HISTORY_NAME
        checkpoint_path = seed_output_dir / CHECKPOINT_NAME
        if history_path.is_file() and checkpoint_path.is_file():
            print(f'Seed {training_seed}: completed artifacts found; skipping.')
            continue
        run_and_log(
            train_command(
                epochs=N_EPOCHS,
                batch_size=BATCH_SIZE,
                project=f'{WANDB_PROJECT}-{VIDEOMAE_VARIANT}-seed-{training_seed}',
                output_dir=seed_output_dir,
                training_seed=training_seed,
                split_seed=SPLIT_SEED,
                temporal_stride=TEMPORAL_STRIDE,
            ),
            f'full_training_seed_{training_seed}.txt',
            log_directory=seed_output_dir,
        )
        verify_training_artifacts(seed_output_dir)
else:
    print('Multi-seed validation skipped.')


Multi-seed validation skipped.


### Multi-seed validation summary


In [16]:
seed_summaries = []
for training_seed in MULTI_SEED_VALUES:
    seed_output_dir = (
        RESULTS_DIR
        if training_seed == SEED
        else MULTI_SEED_ROOT / f'seed_{training_seed}'
    )
    history_path = seed_output_dir / HISTORY_NAME
    split_path = seed_output_dir / SPLITS_NAME
    if not history_path.is_file():
        continue
    seed_summaries.append({
        'seed': training_seed,
        'split_seed': SPLIT_SEED,
        'split_sha256': file_sha256(split_path) if split_path.is_file() else None,
        **selected_validation_metrics(history_path),
    })

if seed_summaries:
    print(json.dumps(seed_summaries, indent=2))
    if len(seed_summaries) > 1:
        import statistics
        accuracies = [item['val_accuracy'] for item in seed_summaries]
        print('Mean selected validation accuracy:', statistics.mean(accuracies))
        print('Validation accuracy standard deviation:', statistics.pstdev(accuracies))
        split_hashes = {item['split_sha256'] for item in seed_summaries}
        if len(split_hashes) != 1:
            raise RuntimeError(
                'Multi-seed runs did not use one identical split file: '
                f'{split_hashes}'
            )
else:
    print('No completed multi-seed histories found yet.')


No completed multi-seed histories found yet.


## 16. Optional temporal-stride ablation


In [17]:
if RUN_TEMPORAL_STRIDE_ABLATION:
    stride_output_dir = (
        PROJECT_ROOT
        / (
            f'results_videomaev2_v7_{VIDEOMAE_VARIANT}_'
            f'stride{TEMPORAL_ABLATION_STRIDE}_seed{SEED}'
        )
    )
    run_and_log(
        train_command(
            epochs=N_EPOCHS,
            batch_size=BATCH_SIZE,
            project=(
                f'{WANDB_PROJECT}-{VIDEOMAE_VARIANT}-'
                f'stride-{TEMPORAL_ABLATION_STRIDE}'
            ),
            output_dir=stride_output_dir,
            training_seed=SEED,
            split_seed=SPLIT_SEED,
            temporal_stride=TEMPORAL_ABLATION_STRIDE,
        ),
        f'full_training_stride_{TEMPORAL_ABLATION_STRIDE}.txt',
        log_directory=stride_output_dir,
    )
    verify_training_artifacts(stride_output_dir)
    stride_history = stride_output_dir / HISTORY_NAME
    if stride_history.is_file():
        print(
            'Stride-ablation validation metrics:',
            selected_validation_metrics(stride_history),
        )
else:
    print('Temporal-stride ablation skipped.')


Temporal-stride ablation skipped.


## 17. Final held-out test evaluation

Keep `RUN_EVALUATION = False` until the backbone, hyperparameters, and training seed protocol have been selected using validation only. Enabling this cell evaluates the exact saved split and checkpoint once.


In [18]:
RUN_EVALUATION = True
if RUN_EVALUATION:
    checkpoint = find_artifact(CHECKPOINT_NAME)
    split_file = find_artifact(SPLITS_NAME)
    if checkpoint is None or split_file is None:
        raise FileNotFoundError(
            'Final evaluation requires best_model_wts.pt and splits.json '
            f'in {RESULTS_DIR}'
        )

    evaluation_command = [
        sys.executable, 'run.py',
        '--frame_dir', str(DATASET_DIR),
        '--mode', 'eval',
        '--output_dir', str(RESULTS_DIR),
        '--split_file', str(split_file),
        '--ckpt', str(checkpoint),
        '--split_protocol', SPLIT_PROTOCOL,
        '--official_split_number', str(OFFICIAL_SPLIT_NUMBER),
        '--architecture', ARCHITECTURE,
        '--n_classes', str(N_CLASSES),
        '--fr_per_vid', str(FRAMES_PER_VIDEO),
        '--image_size', str(IMAGE_SIZE),
        '--temporal_stride', str(TEMPORAL_STRIDE),
        '--val_clips', str(VAL_CLIPS),
        '--test_clips', str(TEST_CLIPS),
        '--flip_tta', str(FLIP_TTA).lower(),
        '--batch_size', str(BATCH_SIZE),
        '--eval_batch_size', str(EVAL_BATCH_SIZE),
        '--eval_clip_chunk_size', str(EVAL_CLIP_CHUNK_SIZE),
        '--workers', str(NUM_WORKERS),
        '--pretrained', 'false',
        '--gradient_checkpointing', 'false',
        '--drop_path_rate', str(DROP_PATH_RATE),
        '--layer_decay', str(LAYER_DECAY),
        '--head_init_scale', str(HEAD_INIT_SCALE),
        '--freeze_batch_norm', 'false',
        '--dropout', str(DROPOUT),
        '--label_smoothing', str(LABEL_SMOOTHING),
        '--amp', str(USE_AMP).lower(),
        '--fingerprint_split_audit', str(FINGERPRINT_SPLIT_AUDIT).lower(),
        '--seed', str(SEED),
        '--wandb_mode', 'disabled',
    ]
    run_and_log(evaluation_command, 'evaluation_output.txt')
else:
    print(
        'Final test evaluation skipped. This is intentional while tuning; '
        'training and checkpoint selection use validation only.'
    )


Running: /usr/bin/python3 run.py --frame_dir /content/hmdb51_data/HMDB51 --mode eval --output_dir /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42 --split_file /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42/splits.json --ckpt /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42/best_model_wts.pt --split_protocol official --official_split_number 1 --architecture videomaev2_vit_s_distilled --n_classes 51 --fr_per_vid 16 --image_size 224 --temporal_stride 4 --val_clips 10 --test_clips 10 --flip_tta true --batch_size 2 --eval_batch_size 1 --eval_clip_chunk_size 1 --workers 2 --pretrained false --gradient_checkpointing false --drop_path_rate 0.1 --layer_decay 0.9 --head_init_scale 0.001 --freeze_batch_norm false --dropout 0.35 --label_smoothing 0.1 --amp true --fingerprint_split_audit true --seed 42 --wandb_mode disabled
Loaded split sizes: train=3033, val=536, test=1530
Loaded split integrity audit passed: {'train_videos': 3033, 'va

## 18. Optional Pylint report


In [19]:
if RUN_PYLINT:
    source_files = [
        'models.py', 'run.py', 'train.py', 'test.py', 'utils.py',
        'video_datasets.py', 'videomaev2.py', 'preflight_videomaev2.py',
        'run_training.py',
    ]
    pylint_log = RESULTS_DIR / 'pylint_output.txt'
    with pylint_log.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            [sys.executable, '-m', 'pylint', *source_files],
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
        status = process.wait()
    print('Pylint exit status:', status)
    print('Pylint report:', pylint_log)
else:
    print('Pylint skipped.')


Pylint skipped.


## 19. Package saved results


In [20]:
import zipfile

verify_training_artifacts()

print('Files retained in the primary Google Drive results directory:')
for path in sorted(RESULTS_DIR.rglob('*')):
    if path.is_file():
        print(f'  {path.relative_to(RESULTS_DIR)} ({path.stat().st_size / 1024:.1f} KB)')

bundle_path = (
    PROJECT_ROOT
    / f'hmdb51_colab_results_videomaev2_v7_{VIDEOMAE_VARIANT}.zip'
)
roots_to_package = [RESULTS_DIR]
if MULTI_SEED_ROOT.exists():
    roots_to_package.append(MULTI_SEED_ROOT)
stride_dir = (
    PROJECT_ROOT
    / (
        f'results_videomaev2_v7_{VIDEOMAE_VARIANT}_'
        f'stride{TEMPORAL_ABLATION_STRIDE}_seed{SEED}'
    )
)
if stride_dir.exists():
    roots_to_package.append(stride_dir)

with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for root in roots_to_package:
        for path in root.rglob('*'):
            if path.is_file():
                archive.write(path, path.relative_to(PROJECT_ROOT))

print('Results bundle:', bundle_path)


Artifact ready: /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42/best_model_wts.pt
Artifact ready: /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42/splits.json
Artifact not found: training_history.json
Artifact ready: /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42/run_config.json
Artifact ready: /content/drive/MyDrive/Homework_4/results_videomaev2_v7_small_seed42/official_duplicate_audit.json
Files retained in the primary Google Drive results directory:
  best_model_wts.pt (85597.8 KB)
  classification_report.json (7.0 KB)
  classification_report.txt (3.0 KB)
  evaluation_output.txt (96.3 KB)
  full_training_output.txt (4051.5 KB)
  official_duplicate_audit.json (0.9 KB)
  pretrained_preflight.json (0.5 KB)
  run_config.json (2.8 KB)
  smoke_test_artifacts/best_model_wts.pt (85597.8 KB)
  smoke_test_artifacts/official_duplicate_audit.json (0.9 KB)
  smoke_test_artifacts/run_config.json (2.8 KB)
  smoke_test_artifacts/smoke_t

## Saved outputs

The primary results directory contains the selected model checkpoint, fixed split metadata, full run configuration, training history, duplicate audit, checkpoint preflight report, and streamed logs. The pretrained source checkpoint remains in the separate persistent cache and is not copied into the results bundle.


In [21]:
# Final notebook cell: recover and sync all available metrics to W&B.

from pathlib import Path
import json
import math
import re
import subprocess
import sys


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

# Uses the VideoMAE notebook's RESULTS_DIR when the configuration cells are
# still in memory. Otherwise, it falls back to the default ViT-S directory.
RESULTS = Path(
    globals().get(
        "RESULTS_DIR",
        "/content/drive/MyDrive/Homework_4/"
        "results_videomaev2_v7_small_seed42",
    )
)

VARIANT = str(globals().get("VIDEOMAE_VARIANT", "small"))

BASE_PROJECT = str(
    globals().get(
        "WANDB_PROJECT",
        "hmdb51-videomaev2-v7",
    )
)

PROJECT = (
    BASE_PROJECT
    if BASE_PROJECT.endswith(f"-{VARIANT}")
    else f"{BASE_PROJECT}-{VARIANT}"
)

ENTITY = globals().get("WANDB_ENTITY") or None

# Leave this as None to create a separate metrics-recovery run.
#
# To append to an existing interrupted W&B run, paste the run ID here.
# For example, for:
# https://wandb.ai/my-team/my-project/runs/abc123xyz
#
# use:
# ORIGINAL_WANDB_RUN_ID = "abc123xyz"
ORIGINAL_WANDB_RUN_ID = None


# -----------------------------------------------------------------------------
# Basic validation
# -----------------------------------------------------------------------------

if not RESULTS.is_dir():
    raise FileNotFoundError(
        f"Results directory not found: {RESULTS}\n"
        "Edit RESULTS near the top of this cell if the notebook variables "
        "are no longer in memory."
    )


# Refuse to read metrics while run.py is still modifying the files.
active_processes = subprocess.run(
    [
        "bash",
        "-lc",
        "pgrep -af '[p]ython.*run.py' || true",
    ],
    capture_output=True,
    text=True,
    check=False,
).stdout.strip()

if active_processes:
    raise RuntimeError(
        "A run.py training or evaluation process is still active. "
        "Stop it before syncing:\n"
        f"{active_processes}"
    )


# -----------------------------------------------------------------------------
# File helpers
# -----------------------------------------------------------------------------

def read_json(filename):
    """Read a JSON artifact if it exists; otherwise return None."""
    path = RESULTS / filename

    if not path.is_file():
        return None

    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Warning: could not read {path.name}: {exc}")
        return None


def finite_number(value):
    """Convert a value to a finite float, or return None."""
    try:
        converted = float(value)
    except (TypeError, ValueError):
        return None

    return converted if math.isfinite(converted) else None


# -----------------------------------------------------------------------------
# Recover per-epoch history
# -----------------------------------------------------------------------------

# This matches the epoch summary printed by train.py:
#
# Epoch 023: train loss=..., train acc=...%,
#            val loss=..., val acc=...%, ...
#
# It is the fallback when interruption prevented training_history.json from
# being written.
epoch_pattern = re.compile(
    r"Epoch\s+(?P<epoch>\d+):\s*"
    r"train loss=(?P<train_loss>[-+0-9.eE]+),\s*"
    r"train acc=(?P<train_accuracy>[-+0-9.eE]+)%,\s*"
    r"val loss=(?P<val_loss>[-+0-9.eE]+),\s*"
    r"val acc=(?P<val_accuracy>[-+0-9.eE]+)%"
)


def parse_training_log(path):
    """Extract all completed epochs from a training text log."""
    rows_by_epoch = {}

    if not path.is_file():
        return []

    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    for match in epoch_pattern.finditer(text):
        epoch = int(match["epoch"])

        rows_by_epoch[epoch] = {
            "epoch": epoch,
            "train_accuracy": (
                float(match["train_accuracy"]) / 100.0
            ),
            "val_accuracy": (
                float(match["val_accuracy"]) / 100.0
            ),
            "train_loss": float(match["train_loss"]),
            "val_loss": float(match["val_loss"]),
        }

    return [
        rows_by_epoch[epoch]
        for epoch in sorted(rows_by_epoch)
    ]


history = read_json("training_history.json")

epoch_rows = []
history_source = "not found"

# Prefer the full-precision JSON file when training ended normally.
if history:
    try:
        train_accuracy = history["accuracy"]["train"]
        val_accuracy = history["accuracy"]["val"]
        train_loss = history["loss"]["train"]
        val_loss = history["loss"]["val"]

        lengths = {
            len(train_accuracy),
            len(val_accuracy),
            len(train_loss),
            len(val_loss),
        }

        if len(lengths) != 1 or not train_accuracy:
            raise ValueError(
                "History arrays are empty or have different lengths."
            )

        epoch_rows = [
            {
                "epoch": index + 1,
                "train_accuracy": float(train_accuracy[index]),
                "val_accuracy": float(val_accuracy[index]),
                "train_loss": float(train_loss[index]),
                "val_loss": float(val_loss[index]),
            }
            for index in range(len(train_accuracy))
        ]

        history_source = str(
            RESULTS / "training_history.json"
        )

    except Exception as exc:
        print(
            "Warning: training_history.json has an invalid "
            f"structure: {exc}"
        )


# When training was interrupted, search the available training logs and use
# whichever contains the greatest number of completed epochs.
if not epoch_rows:
    log_candidates = list(
        RESULTS.glob("*training*.txt")
    )

    primary_log = RESULTS / "full_training_output.txt"

    if primary_log not in log_candidates:
        log_candidates.append(primary_log)

    for log_path in log_candidates:
        parsed_rows = parse_training_log(log_path)

        if len(parsed_rows) > len(epoch_rows):
            epoch_rows = parsed_rows
            history_source = str(log_path)


# -----------------------------------------------------------------------------
# Recover validation-selected checkpoint metrics
# -----------------------------------------------------------------------------

checkpoint_path = RESULTS / "best_model_wts.pt"

best_epoch = None
best_val_accuracy = None
best_val_loss = None

if checkpoint_path.is_file():
    import torch

    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        # Compatibility with older supported PyTorch versions.
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
        )

    if isinstance(checkpoint, dict):
        checkpoint_epoch = checkpoint.get("epoch")

        if checkpoint_epoch is not None:
            # The checkpoint stores a zero-based epoch index.
            best_epoch = int(checkpoint_epoch) + 1

        best_val_accuracy = finite_number(
            checkpoint.get("val_accuracy")
        )

        best_val_loss = finite_number(
            checkpoint.get("val_loss")
        )

    del checkpoint


# Fall back to selecting the best row from the recovered history if the
# checkpoint could not be read.
if best_val_accuracy is None and epoch_rows:
    best_row = max(
        epoch_rows,
        key=lambda row: (
            row["val_accuracy"],
            -row["val_loss"],
        ),
    )

    best_epoch = best_row["epoch"]
    best_val_accuracy = best_row["val_accuracy"]
    best_val_loss = best_row["val_loss"]


# -----------------------------------------------------------------------------
# Recover held-out test and per-class metrics
# -----------------------------------------------------------------------------

test_metrics = read_json("test_metrics.json") or {}

# If evaluation was interrupted after printing the result but before writing
# test_metrics.json, recover the two headline metrics from its output log.
evaluation_output = RESULTS / "evaluation_output.txt"

if not test_metrics and evaluation_output.is_file():
    evaluation_text = evaluation_output.read_text(
        encoding="utf-8",
        errors="replace",
    )

    test_loss_match = re.search(
        r"Test loss:\s*([-+0-9.eE]+)",
        evaluation_text,
    )

    test_accuracy_match = re.search(
        r"Test accuracy:\s*([-+0-9.eE]+)%",
        evaluation_text,
    )

    if test_loss_match:
        test_metrics["test_loss"] = float(
            test_loss_match[1]
        )

    if test_accuracy_match:
        test_metrics["test_accuracy"] = (
            float(test_accuracy_match[1]) / 100.0
        )


classification_report = (
    read_json("classification_report.json") or {}
)

run_config = read_json("run_config.json") or {}


if (
    not epoch_rows
    and best_val_accuracy is None
    and not test_metrics
):
    raise FileNotFoundError(
        "No recoverable metrics were found. Expected at least "
        "one of training_history.json, full_training_output.txt, "
        "best_model_wts.pt, or test_metrics.json."
    )


# -----------------------------------------------------------------------------
# Build W&B summary metrics
# -----------------------------------------------------------------------------

summary = {
    "recovered/epochs_completed": len(epoch_rows),
    "recovered/history_source": history_source,
}


if best_epoch is not None:
    summary["best/epoch"] = best_epoch

if best_val_accuracy is not None:
    summary["best/val_accuracy"] = best_val_accuracy

if best_val_loss is not None:
    summary["best/val_loss"] = best_val_loss


if epoch_rows:
    final_row = epoch_rows[-1]

    summary.update(
        {
            "final/epoch": final_row["epoch"],
            "final/train_accuracy": (
                final_row["train_accuracy"]
            ),
            "final/val_accuracy": (
                final_row["val_accuracy"]
            ),
            "final/train_loss": (
                final_row["train_loss"]
            ),
            "final/val_loss": (
                final_row["val_loss"]
            ),
            "final/generalization_gap": (
                final_row["train_accuracy"]
                - final_row["val_accuracy"]
            ),
        }
    )


for source_name, wandb_name in (
    ("test_accuracy", "test/accuracy"),
    ("test_loss", "test/loss"),
):
    value = finite_number(
        test_metrics.get(source_name)
    )

    if value is not None:
        summary[wandb_name] = value


# The sklearn classification report also contains an overall accuracy field.
if (
    "test/accuracy" not in summary
    and finite_number(
        classification_report.get("accuracy")
    ) is not None
):
    summary["test/accuracy"] = finite_number(
        classification_report["accuracy"]
    )


for report_name, prefix in (
    ("macro avg", "test/macro"),
    ("weighted avg", "test/weighted"),
):
    values = classification_report.get(
        report_name,
        {},
    )

    if not isinstance(values, dict):
        continue

    for source_name, metric_name in (
        ("precision", "precision"),
        ("recall", "recall"),
        ("f1-score", "f1"),
        ("support", "support"),
    ):
        value = finite_number(
            values.get(source_name)
        )

        if value is not None:
            summary[
                f"{prefix}/{metric_name}"
            ] = value


# -----------------------------------------------------------------------------
# Install and initialize W&B
# -----------------------------------------------------------------------------

try:
    import wandb
except ImportError:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "wandb>=0.17",
        ],
        check=True,
    )

    import wandb


# This is a no-op when the runtime is already authenticated.
wandb.login()


run_config.update(
    {
        "results_dir": str(RESULTS),
        "posthoc_metrics_sync": True,
    }
)


init_arguments = {
    "project": PROJECT,
    "entity": ENTITY,
}


if ORIGINAL_WANDB_RUN_ID:
    # Append to the original interrupted run.
    init_arguments.update(
        {
            "id": ORIGINAL_WANDB_RUN_ID,
            "resume": "allow",
        }
    )

else:
    # Safer default: retain the original run and create a separate archive run.
    init_arguments.update(
        {
            "name": (
                f"{RESULTS.name}-metrics-recovery"
            ),
            "job_type": "metrics-recovery",
            "group": RESULTS.name,
            "tags": [
                "posthoc-metrics",
                "videomaev2",
                VARIANT,
            ],
            "config": run_config,
        }
    )


# -----------------------------------------------------------------------------
# Upload metrics
# -----------------------------------------------------------------------------

with wandb.init(**init_arguments) as run:
    # Also update the configuration when resuming an existing run.
    run.config.update(
        run_config,
        allow_val_change=True,
    )

    # A custom x-axis avoids moving W&B's internal global step backwards when
    # this cell appends recovered history to an existing interrupted run.
    run.define_metric(
        "recovered/epoch"
    )

    for metric_name in (
        "recovered/train_accuracy",
        "recovered/val_accuracy",
        "recovered/train_loss",
        "recovered/val_loss",
    ):
        run.define_metric(
            metric_name,
            step_metric="recovered/epoch",
        )


    # Per-epoch train and validation curves.
    for row in epoch_rows:
        run.log(
            {
                "recovered/epoch": (
                    row["epoch"]
                ),
                "recovered/train_accuracy": (
                    row["train_accuracy"]
                ),
                "recovered/val_accuracy": (
                    row["val_accuracy"]
                ),
                "recovered/train_loss": (
                    row["train_loss"]
                ),
                "recovered/val_loss": (
                    row["val_loss"]
                ),
            }
        )


    # A table containing the exact recovered epoch values.
    if epoch_rows:
        epoch_table = wandb.Table(
            columns=[
                "epoch",
                "train_accuracy",
                "val_accuracy",
                "train_loss",
                "val_loss",
            ],
            data=[
                [
                    row["epoch"],
                    row["train_accuracy"],
                    row["val_accuracy"],
                    row["train_loss"],
                    row["val_loss"],
                ]
                for row in epoch_rows
            ],
        )

        run.log(
            {
                "recovered/epoch_metrics": (
                    epoch_table
                )
            }
        )


    # Per-class precision, recall, F1, and support from final test evaluation.
    excluded_report_rows = {
        "accuracy",
        "macro avg",
        "weighted avg",
        "micro avg",
    }

    class_rows = [
        [
            class_name,
            finite_number(
                values.get("precision")
            ),
            finite_number(
                values.get("recall")
            ),
            finite_number(
                values.get("f1-score")
            ),
            finite_number(
                values.get("support")
            ),
        ]
        for class_name, values
        in classification_report.items()
        if (
            class_name
            not in excluded_report_rows
            and isinstance(values, dict)
        )
    ]


    if class_rows:
        class_table = wandb.Table(
            columns=[
                "class",
                "precision",
                "recall",
                "f1",
                "support",
            ],
            data=sorted(class_rows),
        )

        run.log(
            {
                "test/per_class_metrics": (
                    class_table
                )
            }
        )


    # Store scalar values in run history and pin the authoritative values in
    # the W&B run summary.
    scalar_summary = {
        key: value
        for key, value in summary.items()
        if isinstance(value, (int, float))
    }

    if scalar_summary:
        run.log(scalar_summary)


    for key, value in summary.items():
        run.summary[key] = value


    run_url = run.url


print("W&B sync complete:", run_url)
print("Epochs uploaded:", len(epoch_rows))
print(
    "Best validation accuracy:",
    summary.get(
        "best/val_accuracy",
        "not available",
    ),
)
print(
    "Test accuracy:",
    summary.get(
        "test/accuracy",
        "not available",
    ),
)

best/epoch,▁
best/val_accuracy,▁
best/val_loss,▁
final/epoch,▁
final/generalization_gap,▁
final/train_accuracy,▁
final/train_loss,▁
final/val_accuracy,▁
final/val_loss,▁
recovered/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
+15,...


W&B sync complete: https://wandb.ai/tim-vaught12-johns-hopkins-university/hmdb51-videomaev2-v7-small/runs/akobuxek
Epochs uploaded: 33
Best validation accuracy: 0.9029850746268657
Test accuracy: 0.8450980186462402
